# Style Transfer with DALL·E

Upload one image for the **subject** (what you want depicted) and another
for the **style** (how you want it to look). GPT-5.2 describes each, then
DALL·E 3 generates a new image combining them.

**How to use:**
1. Run the *Setup* cell once (imports + API key).
2. Run the *Helpers* cell once to define the pipeline functions.
3. Run the **Generate** cell whenever you want to make a new image.
   It will prompt you to upload two files: first the content image, then the style image.


In [ ]:
# === Setup: imports and OpenAI client ===
# Run this cell once per session.

import os
import base64
from openai import OpenAI
from PIL import Image as PILImage
from IPython.display import Image, display
from google.colab import files

# Paste your OpenAI API key here (or load it from an env var / Colab secret).
OPENAI_API_KEY = ""

client = OpenAI(api_key=OPENAI_API_KEY)


In [2]:
# === Helpers: image prep + captioning + style transfer ===
# Run this cell once per session. You don't need to touch anything here
# unless you want to tweak prompts, models, or image sizes.

def prepare_image(image_path, output_path):
    """Convert an uploaded image to a PNG (<4 MB, RGB) and return a data URL.

    GPT-5.2 needs a URL to see the image. Since the user's file is local,
    we encode it as a base64 data URL so we don't have to host it anywhere.
    """
    img = PILImage.open(image_path)

    # Drop alpha / palette modes — APIs expect RGB PNGs.
    if img.mode in ("RGBA", "P"):
        img = img.convert("RGB")
    img.save(output_path, "PNG", optimize=True)

    # Shrink iteratively until under 4 MB (a safe limit for the vision API).
    while os.path.getsize(output_path) / (1024 * 1024) >= 4:
        img = img.resize(
            (int(img.width * 0.9), int(img.height * 0.9)),
            PILImage.Resampling.LANCZOS,
        )
        img.save(output_path, "PNG", optimize=True)

    # Encode as a data URL so it can be passed directly into the chat API.
    with open(output_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    return f"data:image/png;base64,{b64}"


def get_image_caption(client, image_url):
    """Ask GPT-5.2 to describe the *subject matter* of an image."""
    response = client.chat.completions.create(
        model="gpt-5.2",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text":
                    "Describe this image in detail, focusing on the key elements, "
                    "composition, and subject matter. Be specific but concise."},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        }],
    )
    return response.choices[0].message.content


def get_style_description(client, image_url):
    """Ask GPT-5.2 to describe the *artistic style* of an image (not the subject)."""
    response = client.chat.completions.create(
        model="gpt-5.2",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text":
                    "Describe the artistic style of this image, including colors, "
                    "techniques, mood, and visual characteristics. Focus on HOW "
                    "it's made rather than WHAT it shows."},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        }],
    )
    return response.choices[0].message.content


def create_styled_image(client, content_description, style_description):
    """Use DALL·E 3 to render `content_description` in the given `style_description`."""
    prompt = (
        f"Create an image showing {content_description}. "
        f"Apply the following artistic style: {style_description}. "
        "Maintain the original subject matter while fully adopting the reference style."
    )
    response = client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        n=1,
        size="1024x1024",
        quality="standard",
    )
    return response.data[0].url


def _upload_one(label):
    """Prompt for a single upload and return its filename."""
    print(f"Please upload your {label} image...")
    uploaded = files.upload()
    return next(iter(uploaded))


def generate_styled_image():
    """End-to-end pipeline: upload content + style → caption each → final image."""
    # 1. Get both images from the user. Two separate prompts so it's clear which is which.
    content_file = _upload_one("CONTENT (the subject you want depicted)")
    style_file   = _upload_one("STYLE (the look you want applied)")

    # 2. Normalize each image and get a data URL we can pass to GPT-5.2.
    content_url = prepare_image(content_file, "content_image.png")
    style_url   = prepare_image(style_file,   "style_image.png")

    # 3. Describe each image with GPT-5.2 — subject for content, aesthetic for style.
    content_desc = get_image_caption(client, content_url)
    style_desc   = get_style_description(client, style_url)
    print(f"Content Desc: {content_desc}\n")
    print(f"Style Desc: {style_desc}\n")

    # 4. Ask DALL·E 3 to render the content in the style.
    final_url = create_styled_image(client, content_desc, style_desc)

    # 5. Show the final image inline.
    print("Final styled image (DALL·E 3):")
    display(Image(url=final_url))

    return final_url


## Generate a new image

Run the cell below to create a new styled image. You'll be asked to upload
**two files in order**: first the content image, then the style image.

In [4]:
# === Generate ===
# This is the only cell you need to re-run to make a new image.
final_image_url = generate_styled_image()


Please upload your CONTENT (the subject you want depicted) image...


Saving images.jpg to images.jpg
Please upload your STYLE (the look you want applied) image...


Saving generated_vaporwave_20250522_091529_0.png to generated_vaporwave_20250522_091529_0 (1).png
Content Desc: A gray-brown tabby cat with dark stripes is stretched out lengthwise along a narrow white ledge or shelf. The cat lies on its side with its body extended, one foreleg reaching forward and the other relaxed, while its tail drapes off the left edge. Its head is lifted and turned toward the camera, giving a calm, slightly alert expression with half-open eyes and upright ears. The scene is minimal and bright, dominated by white walls and the ledge, with the cat centered horizontally and framed by the clean, uncluttered background.

Style Desc: The image is rendered in a glossy, hyper-saturated digital illustration style that blends **neon pop** with **dreamlike/synthwave color grading**.

- **Color palette:** Intensely vivid, candy-like hues—electric cyan, deep cobalt, magenta, hot pink, and bright lemon yellow—pushed to near “glow” levels. The background uses smooth gradients (t